# Handwritten Digit Recognizer - PyTorch Baseline

欢迎来到本仓库！这份 Notebook 演示了如何使用 PyTorch 从零构建一个高准确率的卷积神经网络 (CNN) 来识别手写数字 (MNIST)。

代码集成了**数据增强 (Data Augmentation)** 和**动态学习率调度 (ReduceLROnPlateau)**，在验证集上可以轻松达到 99% 以上的准确率。

**如何在本地运行此代码：**
1. 确保您的电脑上安装了 Python 3 和必要的库：`pytorch`, `torchvision`, `pandas`, `numpy`, `matplotlib`。
2. 确保本 Notebook 同目录下存在 `train.csv` 和 `test.csv` 数据集文件。
3. 如果您有支持 CUDA 的 NVIDIA 显卡，代码会自动调用 GPU 加速训练；如果没有，也会自动使用 CPU 完美运行。

## 1. 导入必要的库和设置环境

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torchvision.transforms as transforms
from PIL import Image

# 固定随机种子以保证结果可复现
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

# 检测是否可用 GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"当前使用的计算设备: {device}")

## 2. 数据加载与自定义 Dataset

默认从当前文件夹 (`./`) 读取 `train.csv` 和 `test.csv`。

In [ ]:
# 默认数据目录为当前文件夹
DATA_DIR = './'

print(f"数据读取路径: {DATA_DIR}")

# 读取 CSV
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f"训练集形状: {train_df.shape}")
print(f"测试集形状: {test_df.shape}")

# 自定义 Dataset
class DigitDataset(Dataset):
    def __init__(self, dataframe, transform=None, is_test=False):
        self.dataframe = dataframe
        self.transform = transform
        self.is_test = is_test
        
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        if self.is_test:
            # 测试集没有 label 列
            pixels = self.dataframe.iloc[idx].values
            label = -1
        else:
            pixels = self.dataframe.iloc[idx, 1:].values
            label = self.dataframe.iloc[idx, 0]
            
        # 转换为 28x28 的形状, 采用 uint8 类型准备转为 PIL 图像
        img_np = pixels.reshape(28, 28).astype(np.uint8)
        
        # 将 numpy 数组转为 PIL Image，这样才能使用 torchvision.transforms 进行高级数据增强
        img_pil = Image.fromarray(img_np)
        
        if self.transform:
            img_tensor = self.transform(img_pil)
        else:
            # 如果没有提供 transform，默认转为 Tensor
            img_tensor = transforms.ToTensor()(img_pil)
            
        return img_tensor, label

## 3. 设置数据增强流水线 (Data Augmentation Pipeline)

使用 `torchvision.transforms` 进行实时的随机变换。这是防止过拟合、冲击极高准确率的核心操作！

In [ ]:
# 训练集：包含旋转、平移、缩放，最后转为 Tensor 并标准化 (使用 MNIST 均值和标准差)
train_transform = transforms.Compose([
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# 验证/测试集：不需要数据增强，直接转为 Tensor 并标准化
val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# 划分训练集和验证集 (保留 10% 作为内部验证集)
# 打乱数据
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
val_split_index = int(len(train_df) * 0.9)

train_data = train_df.iloc[:val_split_index]
val_data = train_df.iloc[val_split_index:]

# 实例化 Dataset
train_dataset = DigitDataset(train_data, transform=train_transform)
val_dataset = DigitDataset(val_data, transform=val_test_transform)
test_dataset = DigitDataset(test_df, transform=val_test_transform, is_test=True)

# 创建 DataLoader 
# 注意：在本地 Windows 环境下，num_workers 设置大于 0 可能会引发多进程报错。
# 如果您使用的是 Linux/Mac，可以将其调大以加速数据读取。
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"训练批次数量: {len(train_loader)}")
print(f"验证批次数量: {len(val_loader)}")

## 4. 可视化检查数据增强效果

In [ ]:
# 取出一个 batch
imgs, labels = next(iter(train_loader))

# 可视化前 16 张图
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    # 反标准化以便显示
    img_show = imgs[i].squeeze().numpy() * 0.3081 + 0.1307
    ax.imshow(img_show, cmap='gray')
    ax.set_title(f"Label: {labels[i].item()}")
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. 定义 CNN 模型

我们构建一个包含多个卷积层、Batch Normalization 以及 Dropout 的强力网络。

In [ ]:
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25)
        )
        
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, stride=2),
            nn.Dropout2d(0.25)
        )
        
        self.fc_block = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 10)
        )
        
    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.fc_block(x)
        return x

model = CNNModel().to(device)
print(model)

## 6. 训练与验证循环

集成 `ReduceLROnPlateau`，当遇到准确率瓶颈时自动降低学习率。

In [ ]:
# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.003)

# 定义 ReduceLROnPlateau 调度器
# 当验证集准确率（mode='max'）连续 3 个 epoch (patience=3) 停止上升时，将学习率乘以 0.5 (factor=0.5)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-6)

epochs = 30
best_val_acc = 0.0

# 记录训练过程
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(epochs):
    # --- 训练阶段 ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
    epoch_train_loss = running_loss / total_train
    epoch_train_acc = correct_train / total_train
    
    # --- 验证阶段 ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    epoch_val_loss = val_loss / total_val
    epoch_val_acc = correct_val / total_val
    
    # 将当前的验证集准确率喂给调度器，它会自动判断是否需要降低学习率
    scheduler.step(epoch_val_acc)
    
    # 保存最好的模型权重
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), 'best_single_model.pth')
    
    history['train_loss'].append(epoch_train_loss)
    history['val_loss'].append(epoch_val_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_acc'].append(epoch_val_acc)
    
    print(f"Epoch [{epoch+1}/{epochs}] | LR: {optimizer.param_groups[0]['lr']:.6f} | "
          f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f} | "
          f"Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")
    
print(f"\n训练结束，内部验证集最高准确率: {best_val_acc:.4f}")

## 7. 训练过程可视化

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss over Epochs')

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')
plt.legend()
plt.title('Accuracy over Epochs')
plt.show()

## 8. 推理并生成结果文件

我们加载刚才保存的在验证集上表现最好的模型参数，对测试集进行预测，并将预测结果导出为 CSV 文件。

In [ ]:
# 加载最佳模型权重
model.load_state_dict(torch.load('best_single_model.pth'))
model.eval()

predictions = []

with torch.no_grad():
    for inputs, _ in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        
        # 得到预测类别
        _, predicted = torch.max(outputs, 1)
        predictions.extend(predicted.cpu().numpy())

# 生成提交文件
submission = pd.DataFrame({
    'ImageId': np.arange(1, len(predictions) + 1),
    'Label': predictions
})

submission.to_csv('submission.csv', index=False)
print("✅ 预测完毕！结果已保存至 submission.csv")
submission.head()